# Profiling VMC Runs

This notebook shows the profiling tools that are now available for this repository.

- Use `cProfile` when you want Python-level hotspots.
- Use `snakeviz` to inspect a saved `cProfile` result visually.
- Use `scalene` when you want Python/native time and memory breakdowns.
- Use `jax.profiler` when the slow part is likely inside JAX/XLA execution.

The workload below is intentionally small: a short 1D Ising VMC run with our RBM model and the temporary NetKet backend.

In [ ]:
from pathlib import Path

from performance_profiling_helper import (
    cprofile_vmc_run,
    scalene_command,
    trace_vmc_with_jax_profiler,
)

## Python-level profiling with cProfile

This is the first tool to use when the overhead might come from driver logic, repeated object construction, or notebook/demo orchestration.

In [ ]:
stats_path = Path("cprofile_vmc.prof")
profile_result = cprofile_vmc_run(n_iter=3, output_path=stats_path)
print(profile_result["summary"])
print(f"Saved cProfile stats to: {stats_path.resolve()}")

If you want a visual browser for the saved `cProfile` output, run this in a terminal:

```powershell
snakeviz Balint/demos/cprofile_vmc.prof
```

## JAX tracing

Use this when `cProfile` shows little Python time but the run is still slow. That usually means the cost is in JAX/XLA execution rather than pure Python.

In [ ]:
trace_dir = Path("jax_profile_trace")
history = trace_vmc_with_jax_profiler(trace_dir, n_iter=3)
print(f"Wrote JAX trace to: {trace_dir.resolve()}")
print("Last recorded energy:", history[-1]["energy"])

The trace directory can be opened with TensorBoard's profile tooling or with Perfetto-compatible viewers.

## Scalene

Scalene runs from the terminal. The helper prints a ready-to-run command.

In [ ]:
print(scalene_command())